In [1]:
# 1. PostgreSQL 설치
!apt-get -y -qq update
!apt-get -y -qq install postgresql postgresql-contrib

# 2. 서버 실행
!service postgresql start

# 3. 설치 확인 (버전 체크)
!sudo -u postgres psql -c "SELECT version();"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package logrotate.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../00-logrotate_3.19.0-1ubuntu1.1_amd64.deb ...
Unpacking logrotate (3.19.0-1ubuntu1.1) ...
Selecting previously unselected package netbase.
Preparing to unpack .../01-netbase_6.3_all.deb ...
Unpacking netbase (6.3) ...
Selecting previously unselected package libcommon-sense-perl:amd64.
Preparing to unpack .../02-libcommon-sense-perl_3.75-2build1_amd64.deb ...
Unpacking libcommon-sense-perl:amd64 (3.75-2build1) ...
Selecting previously unselected package libjson-perl.
Preparing to unpack .../03-libjson-perl_4.04000-1_all.deb ...
Unpacking libjson-perl (4.04000-1) ...
Selecting previously unselected package libtypes-serialiser-perl

In [3]:
!pg_isready

/var/run/postgresql:5432 - accepting connections


In [2]:
# 사용자: lion_user / 비밀번호: lion123 / DB명: lion_db 생성
!sudo -u postgres psql -c "CREATE USER lion_user WITH ENCRYPTED PASSWORD 'lion123';"
!sudo -u postgres psql -c "CREATE DATABASE lion_db OWNER lion_user;"
!sudo -u postgres psql -c "GRANT ALL PRIVILEGES ON DATABASE lion_db TO lion_user;"

CREATE ROLE
CREATE DATABASE
GRANT


In [4]:
import psycopg2

# DB 연결 설정
conn = psycopg2.connect(
    dbname="lion_db",
    user="lion_user",
    password="lion123",
    host="localhost"
)
cur = conn.cursor()

# 메인 테이블 생성 (BIGSERIAL + 추론 시간 컬럼 추가)
create_table_query = """
CREATE TABLE IF NOT EXISTS detection_main (
    -- 1. 기본 정보 및 식별자
    id BIGSERIAL PRIMARY KEY,               -- 대규모 데이터를 고려한 BIGSERIAL
    dataset VARCHAR(50) NOT NULL,
    category VARCHAR(50) NOT NULL,
    line VARCHAR(50) NOT NULL,

    -- 2. 추론 시간 데이터 (SLA 및 성능 모니터링용)
    ad_start_time TIMESTAMP NOT NULL,        -- AD 추론 시작 시각 (생성 시각 역할 겸함)
    ad_inference_duration FLOAT,             -- AD 소요 시간 (초)
    llm_start_time TIMESTAMP,                -- LLM 추론 시작 시각
    llm_inference_duration FLOAT,            -- LLM 소요 시간 (초)

    -- 3. 이미지 및 시각화 경로
    image_path TEXT NOT NULL,
    heatmap_path TEXT DEFAULT NULL,
    mask_path TEXT DEFAULT NULL,
    similar_image_path TEXT DEFAULT NULL,

    -- 4. AI 판정 결과
    ad_score FLOAT,
    is_anomaly_AD BOOLEAN,
    is_anomaly_LLM BOOLEAN,

    -- 5. 보고서 데이터
    llm_report JSONB DEFAULT '{}'::jsonb,
    llm_summary JSONB DEFAULT '{}'::jsonb
);

-- 인덱스 추가 (조회 성능 향상)
CREATE INDEX IF NOT EXISTS idx_category ON detection_main(category);
CREATE INDEX IF NOT EXISTS idx_ad_start_time ON detection_main(ad_start_time); -- 시간순 조회를 위한 인덱스
CREATE INDEX IF NOT EXISTS idx_llm_report_gin ON detection_main USING GIN (llm_report);
"""

cur.execute(create_table_query)
conn.commit()
print("✅ Table 'detection_main' with inference metrics created successfully!")

cur.close()
conn.close()

Table 'detection_main' created successfully!


In [5]:
import psycopg2

try:
    conn = psycopg2.connect(
        dbname="lion_db",
        user="lion_user",
        password="lion123",
        host="localhost"
    )
    cur = conn.cursor()

    # 카테고리별 상세 설정을 담는 테이블
    create_metadata_sql = """
    CREATE TABLE IF NOT EXISTS category_metadata (
        category VARCHAR(50) PRIMARY KEY, -- 클래스 카테고리 (PK)
        knowledge JSONB,          -- LLM이 참고할 클래스 별 정보
        threshold FLOAT          -- 클래스별로 다르게 적용할 AD 임계치
    );

    -- 메인 테이블과의 연관 관계를 위해 외래키(FK)를 설정할 수도 있지만,
    -- 코랩 테스트 단계에서는 유연성을 위해 인덱스만 먼저 잡겠습니다.
    CREATE INDEX IF NOT EXISTS idx_meta_category ON category_metadata (category);
    """

    cur.execute(create_metadata_sql)
    conn.commit()
    print("✅ 'category_metadata' 테이블 구축 완료!")

except Exception as e:
    print(f"❌ 에러 발생: {e}")
finally:
    if 'cur' in locals(): cur.close()
    if 'conn' in locals(): conn.close()

✅ 'category_metadata' 테이블 구축 완료!
